# Конспект. Модуль 6: Мост к современным реализациям — зачем появились XGBoost/LightGBM/CatBoost

## 1. Зачем это нужно и как это связано с предыдущими модулями

Модули 3–5 дали нам полностью рабочий алгоритм градиентного бустинга («ванильный» GBM в духе оригинальной статьи Фридмана 2001 года): считаем псевдо-остатки как антиградиент, обучаем на них дерево, добавляем с шагом `η`, контролируем переобучение через `subsample` и раннюю остановку. Этот алгоритм **действительно работает** — вы вручную прогнали его в Модуле 4 и увидели, как убывает ошибка.

Тем не менее, именно этот «ванильный» вариант **почти никогда не используется в продакшене напрямую**. Вместо него индустрия использует XGBoost, LightGBM или CatBoost — и в контексте вашей вакансии (Т-Банк, ML-инженер) явно указано: LightGBM — обязательное требование. Этот модуль объясняет **зачем** между ванильным GBM и, например, LightGBM (Модуль 7) вообще потребовался целый дополнительный слой инженерных и математических решений, и разбирает **тот самый мост** — идеи, впервые собранные вместе в XGBoost, на которых так или иначе базируются все современные реализации.

**Важная оговорка по структуре курса:** этот модуль **не** является полным описанием XGBoost как библиотеки (мы не будем разбирать XGBoost так же глубоко, как LightGBM в Модуле 7 или CatBoost в Модуле 8) — он именно **мост**, разбирающий общие принципы, на которые опираются все три современные библиотеки, прежде чем мы пойдём вглубь конкретно LightGBM и CatBoost.

## 2. Три проблемы ванильного GBM

Прежде чем смотреть на решения, чётко сформулируем, что именно не устраивает в алгоритме из Модулей 3–5 при работе на реальных, больших данных (вроде вашего IEEE-CIS с ~590K строк и 150+ признаками в проекте FraudGuard).

### 2.1. Проблема скорости: точный перебор порогов

Вспомните раздел 5 Модуля 1: CART на каждом узле дерева перебирает **все** уникальные значения **каждого** признака как кандидатов на порог разбиения, что требует сортировки — сложность порядка `O(M · N log N)` на уровень дерева, где `M` — число признаков, `N` — число объектов. При `N≈590 000` и `M≈150`, повторённая **на каждом узле каждого из сотен деревьев ансамбля**, эта операция становится вычислительно неподъёмной — счёт идёт на часы или дни вместо минут.

### 2.2. Проблема регуляризации: контроль сложности только «снаружи»

В Модулях 1 и 5 мы контролировали переобучение исключительно **гиперпараметрами**, ограничивающими рост дерева или число итераций (`max_depth`, `min_samples_leaf`, `learning_rate`, ранняя остановка) — это всё **эвристические, внешние** ограничения. Сама **целевая функция**, которую дерево оптимизирует при выборе разбиения (Gini/Entropy/MSE — Модуль 1), **не содержит штрафа за сложность** — дерево жадно ищет максимальное падение impurity, не «зная», что за это придётся заплатить переобучением; вся ответственность за контроль сложности лежит на пользователе, который заранее выставляет ограничивающие гиперпараметры.

Сравните с линейной регрессией и Ridge/Lasso-регуляризацией (Неделя 4) — там штраф `λ·||w||²` встроен **прямо в оптимизируемую функцию**: модель сама, в процессе оптимизации, взвешивает выигрыш в точности против штрафа за сложность. У ванильного GBM такого встроенного механизма нет вовсе.

### 2.3. Проблема категориальных признаков и пропусков

CART (и, соответственно, ванильный GBM поверх него) в реализации sklearn ожидает уже закодированные числа — категориальные признаки нужно вручную кодировать (One-Hot/Ordinal, Неделя 5), а пропуски — вручную импутировать (`SimpleImputer` или вручную выставленная константа вроде `-999`, как вы делали в `FraudGuard` на Дне 2). Оба решения **теряют информацию или добавляют шум**: one-hot на высококардинальных признаках раздувает размерность (Неделя 5), а произвольная константа-заполнитель (`-999`) заставляет дерево искать «искусственный» порог вокруг выброса вместо честной обработки пропуска как отдельного случая.

## 3. XGBoost как мост: регуляризованная целевая функция

Первое и самое концептуально важное нововведение XGBoost — **явный штраф за сложность дерева прямо внутри оптимизируемой функции**, а не только через внешние гиперпараметры.

Для дерева `f`, определяемого структурой разбиений и весами листьев `w_1, ..., w_T` (`T` — число листьев), штраф за сложность:

In [ ]:
Ω(f) = γ·T + (1/2)·λ·Σ(j=1..T) w_j²

- `γ·T` — штраф просто **за число листьев** (аналог `min_samples_split`/`min_impurity_decrease` из Модуля 1, но теперь встроенный в саму оптимизируемую величину, а не отдельное правило остановки).
- `(1/2)·λ·Σw_j²` — **L2-регуляризация на веса листьев**, буквально та же идея Ridge-регрессии (Неделя 4), только применённая не к весам линейной модели, а к предсказаниям в листьях дерева.

Полная целевая функция на итерации `t` (добавляем регуляризацию к сумме потерь по всем объектам, суммируя штраф по всем уже построенным + новому дереву — но так как предыдущие деревья зафиксированы, для оптимизации нового дерева важен только его собственный `Ω(f_t)`):

In [ ]:
Obj_t = Σ(i=1..N) l(y_i, F_{t-1}(x_i) + f_t(x_i)) + Ω(f_t)

## 4. Разложение Тейлора второго порядка — от градиента к Ньютоновскому шагу

Это самая математически насыщенная часть модуля — распишем её медленно и по шагам, опираясь на то, что мы уже знаем из Модуля 3.

### 4.1. Напоминание: первый порядок (то, что мы уже делали)

В Модуле 3 мы использовали **только первую производную** (градиент) функции потерь по предсказанию, чтобы построить псевдо-остаток. Формально, это использование **линейного (первого порядка)** приближения функции потерь рядом с текущим предсказанием.

### 4.2. Идея XGBoost: учесть ещё и кривизну (вторую производную)

Вспомните из базового матанализа: **разложение Тейлора** функции `l(F)` рядом с точкой `F_{t-1}` до **второго** порядка (не только первая, но и вторая производная):

In [ ]:
l(F_{t-1} + Δ) ≈ l(F_{t-1}) + l'(F_{t-1})·Δ + (1/2)·l''(F_{t-1})·Δ²

где `Δ` — малое приращение (в нашем случае — это как раз вклад нового дерева `f_t(x_i)`). Первая производная `l'` — это **градиент** (то, что мы уже считали в Модуле 3 как `g_i = ∂l/∂F`). Вторая производная `l''` — это **Гессиан** (обозначим `h_i = ∂²l/∂F²`) — новая для нас величина, которая показывает **кривизну** функции потерь в этой точке (насколько быстро меняется сам градиент).

**Зачем нужна вторая производная — интуиция.** Первая производная говорит «в какую сторону идти», но не говорит «насколько большой шаг безопасен». Вторая производная (кривизна) как раз про это: если функция потерь в этой точке очень «крутая» и быстро меняющаяся (большой `h_i`), нужно двигаться осторожнее (меньшим шагом); если функция плоская (малый `h_i`), можно двигаться увереннее. Это в точности идея **метода Ньютона** в оптимизации — более быстрый и точный аналог обычного градиентного спуска, который использует не только направление, но и локальную кривизну для выбора **оптимального размера** шага, а не произвольного `η`.

Применяя это к каждому объекту `i` (подставляя `Δ = f_t(x_i)` и отбрасывая константное слагаемое `l(F_{t-1}(x_i))`, которое не зависит от `f_t` и потому не влияет на оптимизацию):

In [ ]:
Obj_t ≈ Σ(i=1..N) [ g_i·f_t(x_i) + (1/2)·h_i·f_t(x_i)² ] + γ·T + (1/2)λ·Σ(j=1..T) w_j²

### 4.3. Вывод оптимального веса листа

Сгруппируем сумму по **листьям**, а не по объектам: пусть `I_j` — множество индексов объектов, попавших в лист `j`. Для всех объектов в листе `j` дерево предсказывает одно и то же значение `w_j` (`f_t(x_i) = w_j` для всех `i ∈ I_j`). Введём:

In [ ]:
G_j = Σ(i∈I_j) g_i        (сумма градиентов объектов, попавших в лист j)
H_j = Σ(i∈I_j) h_i        (сумма гессианов объектов, попавших в лист j)

Тогда целевая функция по всему дереву превращается в сумму `T` независимых **квадратичных** выражений (по одному на каждый лист):

In [ ]:
Obj_t = Σ(j=1..T) [ G_j·w_j + (1/2)·(H_j + λ)·w_j² ] + γ·T

Это классическая задача минимизации квадратичной функции одной переменной `a·w² + b·w` (с `a = (1/2)(H_j+λ)`, `b = G_j`), решение которой — школьная формула вершины параболы `w* = -b/(2a)`:

In [ ]:
w_j* = - G_j / (H_j + λ)

Подставив обратно, получаем минимальное значение объектива для **фиксированной структуры дерева**:

In [ ]:
Obj_t* = - (1/2) · Σ(j=1..T) [ G_j² / (H_j + λ) ] + γ·T

**Это и есть знаменитая «структурная оценка» (structure score) XGBoost** — прямой аналог Gini/Entropy/MSE из Модуля 1, но выведенный **строго из функции потерь**, а не подобранный как отдельный эвристический критерий. Величину `-G_j²/(H_j+λ)` для каждого листа можно понимать как «насколько хорошо этот лист описывает данные, с учётом регуляризации» — чем она отрицательнее (то есть чем больше `G_j²/(H_j+λ)`), тем лучше лист (снижает общий объектив).

### 4.4. Формула Gain для сравнения кандидатов разбиения

Когда дерево решает, стоит ли разбивать лист (с суммами `G`, `H`) на левый (`G_L`, `H_L`) и правый (`G_R`, `H_R`), сравнивается объектив **до** и **после** разбиения — разница называется **Gain**:

In [ ]:
Gain = (1/2) · [ G_L²/(H_L+λ) + G_R²/(H_R+λ) - (G_L+G_R)²/(H_L+H_R+λ) ]  -  γ

**Как читать формулу:** первые два слагаемых в скобках — «качество» двух новых листьев по отдельности, третье — «качество», которое было бы, если бы мы **не** разбивали узел (объединённый лист). Разница показывает **чистый выигрыш** от разбиения. Финальное `-γ` — штраф за появление ещё одного листа: **если выигрыш меньше `γ`, разбиение просто не делается** — это математически строгий, встроенный в саму формулу аналог `min_impurity_decrease` из Модуля 1, только теперь принципиально обоснованный (напрямую следует из регуляризованной целевой функции), а не произвольно подобранный порог.

**Главное следствие этой конструкции — общность.** Формула Gain использует только `g_i` и `h_i` — первую и вторую производные функции потерь. Она **не завязана** на конкретный вид `l` (MSE, LogLoss или что-то ещё) — стоит только вычислить `g_i` и `h_i` для новой функции потерь (даже кастомной, придуманной под конкретную бизнес-задачу), и та же самая формула Gain будет работать без изменений. Это принципиально отличается от Модуля 1, где для регрессии и классификации использовались **разные** критерии (MSE vs Gini/Entropy) — XGBoost унифицировал их в одну универсальную формулу через математику самой функции потерь.

## 5. Численная проверка: применяем формулу Gain к нашему датасету из Модуля 4

Возьмём тот же датасет из Модуля 4: `x=[1,2,5,6,9,10]`, `y=[4,6,14,16,29,31]`, `F_0 = 16.667`.

**Важно про обозначения.** В Модуле 3 мы определяли псевдо-остаток `r_i = -∂l/∂F` (антиградиент). Здесь, следуя стандартной нотации XGBoost, `g_i = +∂l/∂F` (сам градиент, **без** минуса) — то есть `g_i = -r_i`. Не перепутайте знак при сверке с предыдущими модулями.

Для MSE `l=(1/2)(y-F)²`: `g_i = F - y_i`, `h_i = ∂²l/∂F² = 1` (константа — вторая производная MSE по `F` всегда равна 1, кривизна не зависит от точки).

In [ ]:
g_1 = 16.667-4 = 12.667    g_2 = 16.667-6 = 10.667
g_3 = 16.667-14 = 2.667    g_4 = 16.667-16 = 0.667
g_5 = 16.667-29 = -12.333  g_6 = 16.667-31 = -14.333
h_i = 1  для всех i

**Проверяем разбиение `x≤7.5`** (то самое, что мы нашли в Модуле 4 через прямой перебор весов MSE):

In [ ]:
G_L = 12.667+10.667+2.667+0.667 = 26.668,  H_L = 4
G_R = -12.333-14.333 = -26.666,            H_R = 2

При `λ=0, γ=0` (без регуляризации, чтобы сравнить «на равных» с невзвешенным MSE-критерием Модуля 1/4):

In [ ]:
Gain = 0.5·[ 26.668²/4 + 26.666²/2 - (26.668-26.666)²/6 ] - 0
     = 0.5·[ 177.80 + 355.54 - ~0 ] = 266.67

**Проверяем альтернативное разбиение `x≤3.5`** (второе по качеству в нашем ручном переборе Модуля 4):

In [ ]:
G_L = 12.667+10.667 = 23.334,  H_L = 2
G_R = 2.667+0.667-12.333-14.333 = -23.332,  H_R = 4

Gain = 0.5·[ 23.334²/2 + 23.332²/4 - ~0 ] = 0.5·[272.24+136.09] = 204.17

**Результат: `Gain(x≤7.5)=266.67 > Gain(x≤3.5)=204.17`** — XGBoost-формула выбирает **тот же самый** лучший сплит (`x≤7.5`), что мы нашли вручную через прямую минимизацию взвешенного MSE в Модуле 4. Это не совпадение: для MSE с `h_i=1` и `λ=0`, формула Gain математически **эквивалентна** обычному критерию уменьшения дисперсии из Модуля 1 — просто выраженному через другую, более общую алгебраическую форму, которая **при этом** обобщается на любую функцию потерь.

**Проверяем формулу оптимального веса листа:**

In [ ]:
w_L* = -G_L/(H_L+λ) = -26.668/4 = -6.667
w_R* = -G_R/(H_R+λ) = 26.666/2 = 13.333

Сравните с Модулем 4: там мы вручную посчитали `h_1(x)=-6.667` для левого листа и `h_1(x)=13.333` для правого, просто как **среднее остатков** в каждой группе. **Значения совпадают в точности.** Это показывает, что «наивный» подход Модуля 4 (среднее остатков как значение листа) — на самом деле частный случай формулы `w* = -G/(H+λ)` при `λ=0` и `h_i=1` (что верно именно для MSE): `-G_j/H_j = -Σg_i/n_j = Σ(y_i-F)/n_j` — это буквально среднее остатков. Общая формула XGBoost **обобщает** этот частный случай на произвольные функции потерь (через `h_i`, которое для LogLoss уже не константа) и добавляет регуляризацию `λ`.

### 5.1. Эффект регуляризации `λ` — численно

Возьмём то же разбиение `x≤7.5`, но теперь `λ=5`:

In [ ]:
w_L* = -26.668/(4+5) = -2.963   (было -6.667 без регуляризации)
w_R* = 26.666/(2+5)  = 3.809    (было 13.333 без регуляризации)

Gain = 0.5·[26.668²/9 + 26.666²/7 - ~0] = 0.5·[79.02+101.58] = 90.30
       (было 266.67 без регуляризации)

**Видно наглядно:** регуляризация **сжимает** значения листьев к нулю (классический эффект Ridge-регуляризации, Неделя 4, только теперь применённый к выходу дерева) и **уменьшает** привлекательность разбиения (Gain упал с 266.67 до 90.30) — если бы у вас был порог `γ` выше 90.30, это разбиение вообще не было бы сделано, хотя без регуляризации оно выглядело очень привлекательно.

### 5.2. Гессиан для LogLoss — красивая связь с вероятностью

Для LogLoss мы уже вывели в Модуле 3: `g_i = p_i - y_i` (где `p_i=σ(F_i)`). Гессиан — вторая производная, берём производную от `g_i=p-y` по `F` ещё раз: так как `dp/dF = p(1-p)` (та же формула производной сигмоиды из Модуля 3), получаем:

In [ ]:
h_i = p_i·(1-p_i)

**Красивое наблюдение** (полезная связка с теорией вероятностей, которую вы сейчас изучаете параллельно): `p(1-p)` — это в точности **дисперсия распределения Бернулли** с параметром `p`. То есть кривизна LogLoss в точке `F` численно равна **статистической неопределённости** предсказания в этой точке! Практическое следствие: объекты, где модель уже **уверена** (`p` близко к 0 или 1) имеют **маленький** гессиан — они меньше влияют на знаменатель `H_j+λ` при вычислении оптимального веса листа, то есть модель естественным образом «меньше прислушивается» к объектам, в которых она и так уверена, и больше — к неопределённым (`p≈0.5`). Это встроенное, математически обоснованное поведение, а не отдельная эвристика.

## 6. Гистограммный (приближённый) поиск сплитов — почему это почти не теряет в качестве

### 6.1. Идея

Вместо перебора **всех** уникальных значений признака (Модуль 1, `O(N log N)` на признак из-за сортировки), значения признака заранее **бинируются** — разбиваются на фиксированное число корзин (например, 255), и кандидатами на порог становятся только **границы корзин**, а не каждое уникальное значение. Построение самих гистограмм (накопление сумм `G` и `H` в каждой корзине) делается за один линейный проход `O(N)` — без сортировки.

### 6.2. Почему потеря качества минимальна

Ключевая интуиция: реальная «оптимальная» граница разбиения почти никогда не обязана совпадать с точностью до отдельного значения — небольшой сдвиг порога (в пределах ширины одной корзины) обычно меняет Gain на пренебрежимо малую величину, **если корзин достаточно много** относительно масштаба шума в данных. При 255 корзинах на признак с диапазоном, скажем, от 0 до 10000, ширина одной корзины — около 40 единиц: найти оптимальный порог с точностью до 40 единиц практически всегда так же хорошо для итогового Gain, как найти его с точностью до 1 единицы, особенно с учётом того, что данные всё равно зашумлены. Выигрыш в скорости (не нужно сортировать, не нужно перебирать тысячи уникальных значений) — огромный, а потеря точности — на практике почти не измерима.

**Забегая вперёд:** именно этот гистограммный подход, доведённый до предела эффективности, — основа скорости LightGBM (Модуль 7), где к нему добавляются ещё две дополнительные оптимизации (GOSS и EFB), делающие библиотеку ещё быстрее.

## 7. Встроенная обработка пропусков (sparsity-aware split finding)

Вместо того чтобы заранее заполнять пропуски константой (как вы делали вручную в Модуле 2 проекта FraudGuard — `-999`), XGBoost решает, **куда направлять** объекты с пропуском в конкретном узле, **как часть самого процесса поиска разбиения**:

1. Для кандидата на разбиение алгоритм считает Gain **дважды**: один раз предполагая, что все объекты с пропущенным значением этого признака идут в **левый** узел, второй раз — что все идут в **правый**.
2. Выбирается тот вариант, который даёт **больший Gain** — это направление запоминается как «направление по умолчанию» именно для этого узла.
3. На инференсе, если у нового объекта значение данного признака пропущено, он автоматически отправляется в запомненное направление по умолчанию.

**Почему это лучше ручной импутации:** алгоритм не навязывает пропуску произвольное значение (вроде `-999`, которое дерево воспринимает как «экстремальный выброс» и вынуждено искать вокруг него порог) — вместо этого он напрямую **учится**, куда логичнее направлять пропуски, основываясь на том, какое направление лучше объясняет целевую переменную в этом конкретном узле дерева. Это прямое развитие идеи Модуля 2 вашего проекта, где вы вручную выбирали константу-заполнитель — здесь та же задача решается принципиально более строго, силами самого алгоритма.

## 8. Сравнительная таблица: XGBoost, LightGBM, CatBoost

| Ось сравнения | XGBoost | LightGBM | CatBoost |
|---|---|---|---|
| Рост дерева | Level-wise (по уровням) | Leaf-wise (по листу с макс. gain) | Symmetric / Oblivious (одно правило на весь уровень) |
| Поиск сплита | Точный или гистограммный (приближённый) | Гистограммный + GOSS (сэмплирование по градиенту) | Гистограммный |
| Категориальные признаки | Нужен ручной энкодинг | Частичная поддержка из коробки | Полная поддержка из коробки (Ordered Target Statistics) |
| Обработка пропусков | Sparsity-aware (раздел 7) | Есть встроенная обработка | Есть встроенная обработка |
| Регуляризация | `γ`, `λ` — явно в целевой функции (разделы 3–4) | Те же идеи + `num_leaves`, `min_data_in_leaf` (Модуль 7) | Встроенная регуляризация через структуру дерева + Ordered Boosting (Модуль 8) |
| Главная сильная сторона | Стабильность, зрелость экосистемы, первым ввёл регуляризованный Newton-boosting | Скорость на больших данных, низкое потребление памяти | Категориальные признаки «из коробки», устойчивость к переобучению |

**Важно понимать эту таблицу правильно:** LightGBM и CatBoost **не отбрасывают** идеи XGBoost (регуляризованный объектив, градиент + гессиан, гистограммный поиск) — они **строятся поверх них**, каждая добавляя свой собственный набор дальнейших инженерных улучшений в своей приоритетной области (скорость — у LightGBM, категории и устойчивость — у CatBoost). Именно поэтому этот модуль называется «мост»: то, что мы разобрали здесь, — общий фундамент, на котором Модули 7 и 8 будут строить более специфичные, продвинутые надстройки.

## 9. Практика: код

### 9.1. Сравнение трёх библиотек на одной задаче

In [ ]:
import time
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

X, y = make_classification(n_samples=50000, n_features=30, n_informative=15,
                            weights=[0.95, 0.05], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

models = {
    "XGBoost": xgb.XGBClassifier(n_estimators=200, random_state=42, eval_metric="logloss"),
    "LightGBM": lgb.LGBMClassifier(n_estimators=200, random_state=42, verbose=-1),
    "CatBoost": cb.CatBoostClassifier(n_estimators=200, random_state=42, verbose=False),
}

for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start

    y_proba = model.predict_proba(X_test)[:, 1]
    pr_auc = average_precision_score(y_test, y_proba)

    print(f"{name:10s} | время: {elapsed:6.2f} сек | PR-AUC: {pr_auc:.4f}")

**Что ожидать:** на параметрах по умолчанию все три библиотеки должны давать **сопоставимое** качество (PR-AUC в пределах небольшого разброса друг от друга) — это подтверждает тезис раздела 8 о том, что все три построены на одном математическом фундаменте. Время обучения, как правило, различается заметнее — LightGBM обычно оказывается быстрее на средних и больших датасетах, что и станет темой отдельного, более пристального разбора в Модуле 7.

### 9.2. Явная проверка формулы Gain руками (numpy)

In [ ]:
import numpy as np

x = np.array([1, 2, 5, 6, 9, 10])
y = np.array([4, 6, 14, 16, 29, 31], dtype=float)
F0 = y.mean()

g = F0 - y          # градиент для MSE: g_i = F - y_i
h = np.ones_like(y) # гессиан для MSE: h_i = 1 (константа)

def gain(mask_left, lam=0.0, gamma=0.0):
    G_L, H_L = g[mask_left].sum(), h[mask_left].sum()
    G_R, H_R = g[~mask_left].sum(), h[~mask_left].sum()
    G, H = G_L + G_R, H_L + H_R
    return 0.5 * (G_L**2/(H_L+lam) + G_R**2/(H_R+lam) - G**2/(H+lam)) - gamma

for threshold in [1.5, 3.5, 5.5, 7.5, 9.5]:
    mask = x <= threshold
    print(f"threshold={threshold}: Gain(λ=0) = {gain(mask):.2f}")

Убедитесь, что порог `7.5` даёт максимальный Gain среди всех кандидатов — сверьте с ручным расчётом раздела 5.

## 10. Частые вопросы на собеседовании

| Вопрос | На что обратить внимание в ответе |
|---|---|
| В чём принципиальное отличие XGBoost от ванильного GBM Фридмана? | Явная регуляризация внутри целевой функции (штраф за число листьев и веса), использование второй производной (Ньютоновский шаг вместо простого градиентного), гистограммный поиск сплитов, встроенная обработка пропусков |
| Зачем использовать вторую производную (гессиан), а не только градиент? | Гессиан даёт информацию о локальной кривизне функции потерь, позволяя точнее (по методу Ньютона) вычислить оптимальный вес листа сразу, а не приближённо через шаг с фиксированным `η`; также формула Gain, выведенная через гессиан, обобщается на любую дважды дифференцируемую функцию потерь, а не только на MSE/LogLoss по отдельности |
| Почему приближённый (гистограммный) поиск сплита почти не теряет в качестве? | При достаточном числе корзин (обычно 255) ширина корзины намного меньше типичного шума в данных — найденный «почти оптимальный» порог даёт Gain, практически неотличимый от точного порога, при этом сложность падает с `O(N log N)` до `O(N)` на признак |
| Как формула Gain связана с критериями Gini/MSE из Модуля 1? | Для MSE (гессиан=1, без регуляризации) формула Gain математически сводится к тому же самому критерию уменьшения взвешенной дисперсии — Gain обобщает классические критерии, а не заменяет их произвольным новым правилом |
| Что происходит с весом листа при увеличении `λ`? | Вес сжимается к нулю (`w*=-G/(H+λ)`, знаменатель растёт) — прямая аналогия с Ridge-регуляризацией, только применённая к предсказаниям в листьях дерева |

## 11. Чек-поинт — попробуйте ответить без подсказок

1. Что даёт использование второй производной (гессиана), а не только градиента?
2. Почему приближённый (гистограммный) поиск сплита почти не теряет в качестве, но сильно выигрывает в скорости?
3. Почему формула Gain, выведенная в этом модуле, для MSE даёт тот же ответ, что и критерий уменьшения дисперсии из Модуля 1?
4. Как регуляризационный параметр `λ` влияет на итоговое значение веса листа, и почему это похоже на то, что вы уже знаете про Ridge-регрессию?
5. Почему подход XGBoost к пропущенным значениям принципиально отличается от заполнения константой (например, `-999`), которое вы использовали в проекте FraudGuard?

## Ответы для самопроверки

<details>
<summary>Раскрыть после того, как попробуете ответить сами</summary>

1. Вторая производная (гессиан) даёт информацию о локальной кривизне функции потерь в текущей точке предсказания — это позволяет вычислить **точный оптимальный** размер шага/значение листа по методу Ньютона (`w*=-G/(H+λ)`), а не приближённо двигаться на произвольный шаг `η`, полагаясь только на направление (как в ванильном GBM Модулей 3–5). Кроме того, формула Gain, построенная на градиенте и гессиане, работает для **любой** дважды дифференцируемой функции потерь единообразно — не нужно придумывать отдельный критерий для регрессии и отдельный для классификации, как в Модуле 1.

2. Потому что при достаточном числе корзин (обычно 255) ширина одной корзины оказывается намного меньше типичного масштаба шума и вариативности в реальных данных — небольшой сдвиг найденного порога в пределах корзины практически не меняет итоговый Gain. При этом отказ от точной сортировки всех уникальных значений снижает сложность с `O(N log N)` до `O(N)` на признак на узел — на больших датасетах это даёт кратный выигрыш в скорости при пренебрежимо малой потере качества.

3. Потому что для MSE вторая производная (гессиан) — это константа `h_i=1` для всех объектов, а при `λ=0` формула `Gain = 0.5·[G_L²/H_L + G_R²/H_R - (G_L+G_R)²/(H_L+H_R)]` с `H_j`, равным просто **числу объектов** в листе (сумма единиц), алгебраически эквивалентна выражению для уменьшения взвешенной дисперсии остатков — той же величине, что мы напрямую вычисляли в Модуле 1 и Модуле 4 через среднеквадратичное отклонение внутри каждого листа. Общая формула XGBoost — обобщение этого частного случая, не замена его другой логикой.

4. Увеличение `λ` увеличивает знаменатель в формуле `w*=-G_j/(H_j+λ)`, что **сжимает** итоговый вес листа ближе к нулю — модель делает более «осторожные», менее экстремальные предсказания в каждом листе. Это концептуально идентично тому, как L2-регуляризация (Ridge, Неделя 4) сжимает веса линейной модели к нулю, штрафуя большие по модулю коэффициенты в самой оптимизируемой функции — здесь тот же принцип применён не к весам признаков линейной модели, а к предсказаниям (весам) листьев дерева.

5. Заполнение константой (`-999`) — это решение, принимаемое **до** обучения модели, вручную и произвольно; дерево видит эту константу как обычное (пусть и экстремальное) значение признака и вынуждено искать вокруг неё какой-то порог, что может исказить реальную структуру данных. XGBoost, напротив, **не заполняет** пропуски заранее — он для каждого узла отдельно проверяет, какое направление (влево или вправо) даёт больший Gain, если отправить туда все объекты с пропуском именно в этом признаке, и запоминает это направление как оптимальное, определённое из данных, а не заданное вручную.

</details>